In [1]:
import os
import importlib
from irisreader import observation
import irisreader
from irisreader.utils.date import to_epoch
from irispy.utils import get_interpolated_effective_area
from datetime import datetime, timedelta
from sunpy.net import Fido
from sunpy.net import attrs as a
from sunpy.timeseries import TimeSeries
import astropy.units as u
from astropy.time import Time
from sunpy.time import parse_time
from tqdm import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from warnings import warn
import h5py
import gc
from astropy import constants as c
import utils_features as uf
import utils_data_prep as utils


global_constants = {
    'fallback_colour': 'grey', #colour to plot if the cluster is not interesting / should not be seen.
    'global_triple_weight_factor': 3, #factor to do the triple weighing with
    'global_save_path': '' #path to save data
}


def save_plot(title, format):
    '''
    Saves the current plot with the given title and format.
    '''
    save_path = global_constants['global_save_path']

    # Check if the file already exists and add a counter if it does
    counter = 1
    original_title = title

    while os.path.exists(f"{save_path}{title}.{format}"):
        title = f"{original_title}_{counter}"
        counter += 1

    plt.savefig(f"{save_path}{title}.{format}", format = format)


    return None


plt.rcParams.update(plt.rcParamsDefault)

plt.rcParams.update({
    'figure.facecolor': 'black',   # Figure background color
    'axes.facecolor': 'black',     # Axes background color
    'axes.edgecolor': 'white',     # Axes edge color
    'axes.labelcolor': 'white',    # X and Y labels
    'xtick.color': 'white',        # X-tick labels
    'ytick.color': 'white',        # Y-tick labels
    'grid.color': 'white',         # Gridlines color
    'text.color': 'white',         # All text (titles, etc.)
    'legend.frameon': True,        # Legend frame (turn it off or set color)
    'legend.facecolor': 'black',   # Legend background
    'legend.edgecolor': 'white',   # Legend edge
    'legend.framealpha': 1.0,      # Fully opaque
    'savefig.facecolor': 'black',  # Set saved figure background color to black
    'savefig.edgecolor': 'black',  # Set saved figure edge color to black
    # 'axes.spines.right': False,    # Disable right border
    # 'axes.spines.top': False,      # Disable top border
    # 'axes.spines.left': True,     # enable left border
    # 'axes.spines.bottom': True,   # enable bottom border
})
print('all figures will be black now.')




all figures will be black now.


In [ ]:
# get GOES data

gc.collect() #collect memory garbage

problem_list = ['']

for obs_id in tqdm(problem_list):
    gc.collect()
    year = obs_id[:4]
    month = obs_id[4:6]
    day = obs_id[6:8]
    pth = f'/sml/iris/{year}/{month}/{day}/{obs_id}'
    obs = observation( pth, keep_null=True )
    obs_start = obs.start_date
    obs_end = obs.end_date
    obs_start = obs_start.replace('T', ' ')
    obs_end = obs_end.replace('T', ' ')
    obs_start = obs_start[:-7]
    obs_end = obs_end[:-7]
    #timerange = a.Time('2015-06-22 17:00', '2015-06-22 21:00')
    timerange = a.Time(obs_start, obs_end)
    hek_query = a.hek.FL & (a.hek.FRM.Name == 'SWPC')
    results = Fido.search(timerange, hek_query)
    hek_results = results['hek']
    print(f'Number of flares: {len(hek_results)}')
    flare_start_times = []
    flare_peak_times = []
    flare_end_times = []
    for flare_number in range(len(hek_results)):
        flares_hek = hek_results[flare_number]
        flare_start_times.append(f'{flares_hek["event_starttime"]}')
        flare_end_times.append(f'{flares_hek["event_endtime"]}')
        flare_peak_times.append(f'{flares_hek["event_peaktime"]}')
    goes_results = Fido.search(timerange, a.Instrument.xrs & a.goes.SatelliteNumber(15))# & a.Resolution("avg1m"))
    downloaded_files = Fido.fetch(goes_results)


    i = 1
    for file in downloaded_files:
        goes_ts = TimeSeries(file)
        fig, ax = plt.subplots()
        goes_ts.plot(axes=ax)
        for time in flare_start_times:
            ax.axvline(parse_time(time).datetime, color='green')
        for time in flare_peak_times:
            ax.axvline(parse_time(time).datetime, color='blue')
        for time in flare_end_times:
            ax.axvline(parse_time(time).datetime, color='red')
        ax.legend(loc=2)
        ax.set_yscale('log')
        plt.title(f'GOES curve: obs_id = {obs_id}, Part {i}/2')
        ax.set_xlim(timerange.start.to_datetime(), timerange.end.to_datetime())
        title = f'GOES_curve_{obs_id}_part_{i}_of_2'
        save_plot(title, 'png')
        save_plot(title, 'pdf')
        plt.show()
        i += 1









In [ ]:
# Clean the data, if before we found that it meets the conditions.



still_to_clean = []


for obs_id in tqdm(still_to_clean, desc='cleaning obses'):
    print(f'Cleaning at the moment: {obs_id}', end='\n\n')
    gc.collect()
    year = obs_id[:4]
    month = obs_id[4:6]
    day = obs_id[6:8]
    pth = f'/sml/iris/{year}/{month}/{day}/{obs_id}'
    obs = observation( pth, keep_null=True )
    raster_MgIIk = obs.raster("Mg II k")
    MgIIk = {'lambda_min' : 2794,
        'lambda_max' : 2806,
        'n_breaks' : 960, #do not modify it! it has to be 960.
        'line' : "Mg II k",
        'field' : "NUV",
        'threshold' : 10
    }
    #prepare obs data with Jonas pipeline:
    obs_MgIIk = utils.Obs_raw_data(obs_id, raster_MgIIk, **MgIIk)
    obs.close()
    # save data as struct
    print('Saving MgIIk')
    type_ = 'AR' #or PF
    obs_MgIIk.save_arr(f'{obs_id}', 'MgIIk', type_)
    print('')
    print('Done cleaning!')



In [ ]:
# save and load goes_data here:



# --------------------------------------------------------------------

gc.collect()
pth = f'/sml/iris/{obs_id[:4]}/{obs_id[4:6]}/{obs_id[6:8]}/{obs_id}'
obs = observation( pth, keep_null=True )
obs_start_ir = obs.start_date
obs_end_ir = obs.end_date
obs_start_ir = obs_start_ir.replace('T', ' ')
obs_end_ir = obs_end_ir.replace('T', ' ')
timerange = a.Time(obs_start_ir, obs_end_ir)
hek_query = a.hek.FL & (a.hek.FRM.Name == 'SWPC')
results = Fido.search(timerange, hek_query)
hek_results = results['hek']
# print(f'Number of flares: {len(hek_results)}')
flare_start_times = []
flare_end_times = []
for flare_number in range(len(hek_results)):
    flares_hek = hek_results[flare_number]
    flare_start_times.append(f'{flares_hek["event_starttime"]}')
    flare_end_times.append(f'{flares_hek["event_endtime"]}')
goes_results = Fido.search(timerange, a.Instrument.xrs & a.goes.SatelliteNumber(15))
downloaded_files = Fido.fetch(goes_results)
goes_ts = TimeSeries(downloaded_files)
times_goes = goes_ts.data.index




In [ ]:

print(type(goes_ts))
times_goes = pd.Series(goes_ts).index  # If you want an index-like structure



df_goes = pd.Series(goes_ts)

# Access the columns
xrsa_data = df_goes['xrsa']
xrsb_data = df_goes['xrsb']

# xrsa_data = goes_ts.data['xrsa']
# xrsb_data = goes_ts.data['xrsb']
length = len(xrsa_data)
flare_start_times_extended = flare_start_times + [0] * (length - len(flare_start_times))
flare_end_times_extended = flare_end_times + [0] * (length - len(flare_end_times))
df_goes = pd.DataFrame({
    'time': times_goes,  # times_goes is the datetime index
    'xrsa': xrsa_data,   # xrsa_data is the first set of GOES data (float)
    'xrsb': xrsb_data,    # xrsb_data is the second set of GOES data (float)
    'flare_start_times': flare_start_times_extended,
    'flare_end_times': flare_end_times_extended
})
path_to_goes_data = f'/sml/jannaschk/GOES_data/'
df_goes.to_csv(f'{path_to_goes_data}goes_data_{obs_id}.csv', index=False)




In [ ]:
# download the rest of ARs:


# obs_id = '20141022_081850_3860261381' #done
# obs_id = '20140906_112339_3820259253'
# obs_id = '20140302_230341_3810263287' # an AR
list_of_ARS = [
    '20150408_045717_3860107054',
    '20150518_143915_3860256971',
    '20150521_185917_3800507454',
    '20150704_100921_3860108354',
    '20150809_061551_3860009180',
    '20150916_181744_3600101141',
    '20151017_003115_3660105403',
    '20151205_225250_3610107123',
    '20161028_212247_3630010059',
]


# --------------------------------------------------------------------

for obs_id in tqdm(list_of_ARS):
    print('current AR:', obs_id)
    gc.collect()
    pth = f'/sml/iris/{obs_id[:4]}/{obs_id[4:6]}/{obs_id[6:8]}/{obs_id}'
    obs = observation( pth, keep_null=True )
    obs_start_ir = obs.start_date
    obs_end_ir = obs.end_date
    obs_start_ir = obs_start_ir.replace('T', ' ')
    obs_end_ir = obs_end_ir.replace('T', ' ')
    timerange = a.Time(obs_start_ir, obs_end_ir)
    hek_query = a.hek.FL & (a.hek.FRM.Name == 'SWPC')
    results = Fido.search(timerange, hek_query)
    hek_results = results['hek']
    # print(f'Number of flares: {len(hek_results)}')
    flare_start_times = []
    flare_end_times = []
    for flare_number in range(len(hek_results)):
        flares_hek = hek_results[flare_number]
        flare_start_times.append(f'{flares_hek["event_starttime"]}')
        flare_end_times.append(f'{flares_hek["event_endtime"]}')
    goes_results = Fido.search(timerange, a.Instrument.xrs & a.goes.SatelliteNumber(15))
    downloaded_files = Fido.fetch(goes_results)
    goes_ts = TimeSeries(downloaded_files)
    times_goes = goes_ts.data.index
    xrsa_data = goes_ts.data['xrsa']
    xrsb_data = goes_ts.data['xrsb']
    length = len(xrsa_data)
    flare_start_times_extended = flare_start_times + [0] * (length - len(flare_start_times))
    flare_end_times_extended = flare_end_times + [0] * (length - len(flare_end_times))
    df_goes = pd.DataFrame({
        'time': times_goes,  # times_goes is the datetime index
        'xrsa': xrsa_data,   # xrsa_data is the first set of GOES data (float)
        'xrsb': xrsb_data,    # xrsb_data is the second set of GOES data (float)
        'flare_start_times': flare_start_times_extended,
        'flare_end_times': flare_end_times_extended
    })
    path_to_goes_data = f'/sml/jannaschk/GOES_data/'
    df_goes.to_csv(f'{path_to_goes_data}goes_data_{obs_id}.csv', index=False)







In [ ]:
# test the obs_id_dict:

# also put obs start and end,
# and the flare time that is relevant for the specific flare we want to look at.


green_flares_to_be_analyzed = [
    "20140329_140938_3860258481", # the third flare is big and on slit. The second is small, on slit. First huge, but next to slit.
    "20140801_172059_3800013190", # only one big flare on slit
    "20140906_112339_3820259253", # third flare on slit, all others off slit
    
    "20141022_081850_3860261381", # third flare on slit (current focus)

    "20150624_111419_3620106017", # second flare big and on slit
    "20151017_104917_3620257130", # first flare big and on slit
]
green_flares_flare_numbers = [
    3,#011 yes
    1,#1 no
    4,#0010 yes
    5,#00100 already done
    2,#01 
    3 #100
]


flares_on_slit_bools = [
    [0, 1, 1], #yes
    [1],
    [0, 0, 1, 0], #yes
    [0, 0, 1, 0, 0], #already done
    [0, 1],
    [1, 0, 0]
]




# Create a dictionary to hold the observation data
obs_id_dict = {
    'obs_id': green_flares_to_be_analyzed,
    'number_of_flares': green_flares_flare_numbers,
    'flares_on_slit': flares_on_slit_bools
}

# Convert the dictionary to a DataFrame
df_obs_id = pd.DataFrame(obs_id_dict)

# Print the DataFrame to verify its contents
print(df_obs_id)

# Save the DataFrame to a CSV file
df_obs_id.to_csv(f'{global_constants["global_save_path"]}green_flares_info_new.csv', index=False)



# Example data
info_dict = {
    'obs_id': green_flares_to_be_analyzed,  # Observation IDs
    'number_of_flares': green_flares_flare_numbers,  # Number of flares
    'flares_on_slit': flares_on_slit_bools  #  Bools of each flares: True = on the slit.
}

df = pd.DataFrame(info_dict)
print(df)

# save the data:
df.to_csv(f'{global_constants["global_save_path"]}green_flares_info.csv', index=False)


In [ ]:
# load the flare infos:

df_loaded = pd.read_csv(f'{global_constants["global_save_path"]}green_flares_info.csv')

print(df_loaded.head())

flare_to_look_at = 3

if df_loaded['flares_on_slit'][flare_to_look_at][0] == True:
    print('True')

print(df_loaded['flares_on_slit'][flare_to_look_at][0])


In [ ]:
# test animation
obs_id  = '20150807_221421_3860259180'

gc.collect()
year = obs_id[:4]
month = obs_id[4:6]
day = obs_id[6:8]
pth = f'/sml/iris/{year}/{month}/{day}/{obs_id}'
obs = observation( pth, keep_null=True )
obs.sji('Si IV').animate()